In [1]:
# ═══════════════════════════════════════════════════════
# 0. 환경 설정
# ═══════════════════════════════════════════════════════
!pip install lightgbm xgboost torch tqdm --quiet

import os, random, re, time, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgbm
from tqdm.auto import tqdm

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path("/content/drive/MyDrive/풍력발전량예측/data")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True, parents=True)

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}


device: cuda
Mounted at /content/drive


In [2]:
# ═══════════════════════════════════════════════════════
# 1. v10 데이터셋
# ═══════════════════════════════════════════════════════
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def dms_to_decimal(s):
    pattern = r"(\d+)°(\d+)'([\d.]+)\"([NS])\s+(\d+)°(\d+)'([\d.]+)\"([EW])"
    m = re.match(pattern, s.strip())
    d1, m1, s1, hemi1, d2, m2, s2, hemi2 = m.groups()
    lat = float(d1) + float(m1) / 60 + float(s1) / 3600
    if hemi1 == "S": lat = -lat
    lon = float(d2) + float(m2) / 60 + float(s2) / 3600
    if hemi2 == "W": lon = -lon
    return lat, lon

def find_nearest_grid(weather_df, coord_df, name_col, lat_col="lat", lon_col="lon"):
    grids = weather_df[["grid_id", "latitude", "longitude"]].drop_duplicates()
    mapping = {}
    for _, row in coord_df.iterrows():
        dists = haversine(row[lat_col], row[lon_col], grids["latitude"], grids["longitude"])
        mapping[row[name_col]] = grids.loc[dists.idxmin(), "grid_id"]
    return mapping

def add_wind_features(df, uv_pairs):
    df = df.copy()
    for u_col, v_col, height in uv_pairs:
        df[f"ws_{height}"] = np.sqrt(df[u_col] ** 2 + df[v_col] ** 2)
        wd_rad = np.arctan2(df[v_col], df[u_col])
        df[f"wd_sin_{height}"] = np.sin(wd_rad)
        df[f"wd_cos_{height}"] = np.cos(wd_rad)
    return df

def build_group_weather(weather_df, group_grid_map, prefix):
    weather_df = weather_df.copy()
    weather_df["forecast_kst_dtm"] = pd.to_datetime(weather_df["forecast_kst_dtm"])
    drop_cols = {"data_available_kst_dtm", "grid_id", "latitude", "longitude"}
    value_cols = [c for c in weather_df.columns if c not in {"forecast_kst_dtm", *drop_cols}]
    frames = []
    for group, grid_id in group_grid_map.items():
        sub = weather_df.loc[weather_df["grid_id"] == grid_id, ["forecast_kst_dtm", *value_cols]].copy()
        sub = sub.rename(columns={c: f"{prefix}_{c}" for c in value_cols})
        sub["kpx_group"] = int(group)
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)

def add_forecast_disagreement(df):
    df = df.copy()
    df["ws_disagreement"] = np.abs(df["ldaps_ws_10m"] - df["gfs_ws_10m"])
    df["wd_agreement"] = df["ldaps_wd_cos_10m"] * df["gfs_wd_cos_10m"] + df["ldaps_wd_sin_10m"] * df["gfs_wd_sin_10m"]
    return df

def add_seasonal_interaction(df):
    df = df.copy()
    month = pd.to_datetime(df["forecast_kst_dtm"]).dt.month
    df["high_variance_season"] = month.isin([1, 2, 3, 4, 5, 7]).astype(int)
    df["ws_x_season"] = df["ldaps_ws_10m"] * df["high_variance_season"]
    df["disagreement_x_season"] = df["ws_disagreement"] * df["high_variance_season"]
    return df

def build_datasets():
    train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
    sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")
    ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
    gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
    ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
    gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")
    train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])
    sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

    raw = pd.read_excel(DATA_DIR / "info.xlsx", sheet_name="info", header=None)
    info = raw.iloc[4:21].copy()
    info.columns = raw.iloc[3]
    info = info.reset_index(drop=True)
    info = info.rename(columns={"좌표(Google)": "coord", "KPX그룹": "kpx_group", "설비용량(MW)": "capacity_mw"})
    info["kpx_group"] = info["kpx_group"].ffill().astype(int)
    coords = info["coord"].apply(dms_to_decimal)
    info["lat"] = coords.apply(lambda x: x[0])
    info["lon"] = coords.apply(lambda x: x[1])
    assert info.groupby("kpx_group")["capacity_mw"].sum().tolist() == [21.6, 21.6, 21.0]

    group_coords = info.groupby("kpx_group")[["lat", "lon"]].mean().reset_index()
    group_coords = group_coords.rename(columns={"kpx_group": "KPX그룹"})
    ldaps_group_grid = find_nearest_grid(ldaps_train, group_coords, "KPX그룹")
    gfs_group_grid = find_nearest_grid(gfs_train, group_coords, "KPX그룹")

    LDAPS_UV_PAIRS = [("heightAboveGround_10_10u", "heightAboveGround_10_10v", "10m")]
    GFS_UV_PAIRS = [
        ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "10m"),
        ("heightAboveGround_80_u", "heightAboveGround_80_v", "80m"),
        ("heightAboveGround_100_100u", "heightAboveGround_100_100v", "100m"),
        ("planetaryBoundaryLayer_0_u", "planetaryBoundaryLayer_0_v", "pbl"),
    ]
    ldaps_train = add_wind_features(ldaps_train, LDAPS_UV_PAIRS)
    ldaps_test = add_wind_features(ldaps_test, LDAPS_UV_PAIRS)
    gfs_train = add_wind_features(gfs_train, GFS_UV_PAIRS)
    gfs_test = add_wind_features(gfs_test, GFS_UV_PAIRS)
    for df in [gfs_train, gfs_test]:
        df["wind_shear_100_10"] = df["ws_100m"] - df["ws_10m"]
        df["wind_shear_80_10"] = df["ws_80m"] - df["ws_10m"]

    ldaps_group_train = build_group_weather(ldaps_train, ldaps_group_grid, "ldaps")
    ldaps_group_test = build_group_weather(ldaps_test, ldaps_group_grid, "ldaps")
    gfs_group_train = build_group_weather(gfs_train, gfs_group_grid, "gfs")
    gfs_group_test = build_group_weather(gfs_test, gfs_group_grid, "gfs")

    train_labels_long = train_labels.melt(id_vars=["kst_dtm"], value_vars=TARGET_COLS,
                                          var_name="kpx_group_col", value_name="target")
    train_labels_long["kpx_group"] = train_labels_long["kpx_group_col"].str.extract(r"(\d+)").astype(int)
    train_labels_long = train_labels_long.rename(columns={"kst_dtm": "forecast_kst_dtm"}).drop(columns=["kpx_group_col"])

    test_base = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
    test_long = pd.concat([test_base.assign(kpx_group=g) for g in [1, 2, 3]], ignore_index=True)

    train_df = train_labels_long.merge(ldaps_group_train, on=["forecast_kst_dtm", "kpx_group"], how="left") \
                                  .merge(gfs_group_train, on=["forecast_kst_dtm", "kpx_group"], how="left")
    test_df = test_long.merge(ldaps_group_test, on=["forecast_kst_dtm", "kpx_group"], how="left") \
                        .merge(gfs_group_test, on=["forecast_kst_dtm", "kpx_group"], how="left")
    assert train_df.shape[0] == train_labels_long.shape[0]

    train_df = add_seasonal_interaction(add_forecast_disagreement(train_df))
    test_df = add_seasonal_interaction(add_forecast_disagreement(test_df))
    weather_cols = [c for c in train_df.columns if c not in ["forecast_kst_dtm", "kpx_group", "target"]]
    print(f"v10 피처 개수: {len(weather_cols)}")
    return train_df, test_df, weather_cols, sample_submission, info

In [3]:
# ═══════════════════════════════════════════════════════
# 2. 공통 유틸 — metric, 편향보정, fold
# ═══════════════════════════════════════════════════════
def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["month"], out["day"], out["hour"] = dt.dt.month, dt.dt.day, dt.dt.hour
    out["dayofweek"] = dt.dt.dayofweek
    out["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)
    out["hour_sin"] = np.sin(2*np.pi*out["hour"]/24); out["hour_cos"] = np.cos(2*np.pi*out["hour"]/24)
    out["month_sin"] = np.sin(2*np.pi*out["month"]/12); out["month_cos"] = np.cos(2*np.pi*out["month"]/12)
    return out

def metric(answer_df, pred_df):
    group_nmae, group_ficr = [], []
    for col in TARGET_COLS:
        actual = answer_df[col].to_numpy(dtype=float); forecast = pred_df[col].to_numpy(dtype=float)
        capacity = CAPACITY_KWH[col]; valid = actual >= capacity * 0.10
        actual, forecast = actual[valid], forecast[valid]
        error_rate = np.abs(forecast - actual) / capacity
        group_nmae.append(np.mean(error_rate))
        unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
        group_ficr.append(np.sum(actual * unit_price) / np.sum(actual * 4.0))
    one_minus_nmae = 1 - np.mean(group_nmae); ficr = np.mean(group_ficr)
    return 0.5 * one_minus_nmae + 0.5 * ficr, one_minus_nmae, ficr

def make_sample_weight(y, capacity, low_weight=0.05):
    return np.where(y >= capacity * 0.10, 1.0, low_weight)

def apply_rolling_smoothing(pred_series, window=3, alpha=0.5):
    rolling = pred_series.rolling(window, center=True, min_periods=1).mean()
    return alpha * pred_series + (1 - alpha) * rolling

def apply_bias_correction_v2(pred, ws, group, bias_map):
    pred = pred.copy()
    if group not in bias_map: return pred
    for interval, bias in bias_map[group].items():
        mask = (ws > interval.left) & (ws <= interval.right)
        pred[mask] = pred[mask] * (1 - bias)
    return pred

def lgbm_params(n_estimators, n_jobs, seed, objective="mae"):
    return dict(n_estimators=n_estimators, max_depth=6, learning_rate=0.03, num_leaves=31,
                min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                random_state=seed, n_jobs=n_jobs, verbose=-1, objective=objective)

def prepare_matrices(tr_g, va_g, weather_cols):
    tr_feat = pd.concat([calendar_features(tr_g["forecast_kst_dtm"]).reset_index(drop=True),
                         tr_g[weather_cols].reset_index(drop=True)], axis=1)
    va_feat = pd.concat([calendar_features(va_g["forecast_kst_dtm"]).reset_index(drop=True),
                         va_g[weather_cols].reset_index(drop=True)], axis=1)
    imputer = SimpleImputer(strategy="median")
    X_tr = pd.DataFrame(imputer.fit_transform(tr_feat), columns=tr_feat.columns)
    X_va = pd.DataFrame(imputer.transform(va_feat), columns=va_feat.columns)
    return X_tr, X_va

FOLDS_Q = [
    {"name": "fold_2024Q1", "train_end": pd.Timestamp("2024-01-01"), "valid_start": pd.Timestamp("2024-01-01 01:00:00"), "valid_end": pd.Timestamp("2024-04-01")},
    {"name": "fold_2024Q2", "train_end": pd.Timestamp("2024-04-01"), "valid_start": pd.Timestamp("2024-04-01 01:00:00"), "valid_end": pd.Timestamp("2024-07-01")},
    {"name": "fold_2024Q3", "train_end": pd.Timestamp("2024-07-01"), "valid_start": pd.Timestamp("2024-07-01 01:00:00"), "valid_end": pd.Timestamp("2024-10-01")},
    {"name": "fold_2024Q4", "train_end": pd.Timestamp("2024-10-01"), "valid_start": pd.Timestamp("2024-10-01 01:00:00"), "valid_end": pd.Timestamp("2025-01-01")},
]
HOLD_FOLD = {"name": "hold_2024full", "train_end": pd.Timestamp("2024-01-01"),
             "valid_start": pd.Timestamp("2024-01-01 01:00:00"), "valid_end": pd.Timestamp("2025-01-01")}
FOLD_LIST = [(f["name"], f) for f in FOLDS_Q] + [(HOLD_FOLD["name"], HOLD_FOLD)]

COARSE_BINS = [0, 11, 13, 100]

def estimate_bias_data(train_df, weather_cols, tr_end, n_estimators, n_jobs,
                       low_weight=0.05, holdout_ratio=0.25, min_fit_days=90):
    out = {}
    for group in [1, 2, 3]:
        capacity = CAPACITY_KWH[f"kpx_group_{group}"]
        g_data = train_df[(train_df["kpx_group"] == group) & (train_df["forecast_kst_dtm"] <= tr_end)]
        g_data = g_data[g_data["target"].notna()]
        if len(g_data) == 0: continue
        first_date = g_data["forecast_kst_dtm"].min()
        available_days = max((tr_end - first_date).days, 1)
        holdout_days = min(int(available_days * holdout_ratio), 180)
        if available_days - holdout_days < min_fit_days:
            holdout_days = max(0, available_days - min_fit_days)
        tr_sub_end = tr_end - pd.Timedelta(days=holdout_days)
        fit_g = g_data[g_data["forecast_kst_dtm"] <= tr_sub_end]
        check_g = g_data[g_data["forecast_kst_dtm"] > tr_sub_end]
        if len(fit_g) == 0 or len(check_g) == 0: continue
        fit_feat = pd.concat([calendar_features(fit_g["forecast_kst_dtm"]).reset_index(drop=True),
                              fit_g[weather_cols].reset_index(drop=True)], axis=1)
        check_feat = pd.concat([calendar_features(check_g["forecast_kst_dtm"]).reset_index(drop=True),
                                check_g[weather_cols].reset_index(drop=True)], axis=1)
        imputer = SimpleImputer(strategy="median")
        X_fit = pd.DataFrame(imputer.fit_transform(fit_feat), columns=fit_feat.columns)
        X_check = pd.DataFrame(imputer.transform(check_feat), columns=check_feat.columns)
        w_fit = make_sample_weight(fit_g["target"].to_numpy(), capacity, low_weight)
        model = lgb.LGBMRegressor(**lgbm_params(n_estimators, n_jobs, SEED))
        model.fit(X_fit, fit_g["target"], sample_weight=w_fit)
        pred_check = model.predict(X_check)
        check_g = check_g.copy()
        check_g["signed_error_rate"] = (pred_check - check_g["target"]) / capacity
        check_g = check_g.loc[check_g["target"] >= capacity * 0.10]
        if len(check_g) == 0: continue
        out[group] = check_g[["ldaps_ws_10m", "signed_error_rate"]]
    return out

def bias_map_from_data(bias_data, bins_by_group):
    bias_map = {}
    for group, df in bias_data.items():
        ws_bin = pd.cut(df["ldaps_ws_10m"], bins=bins_by_group[group])
        bias_map[group] = df.groupby(ws_bin, observed=True)["signed_error_rate"].mean().to_dict()
    return bias_map

In [4]:
# ═══════════════════════════════════════════════════════
# 3. v10TPS 피처 (T+P+S 블록)
# ═══════════════════════════════════════════════════════
def add_physics(df):
    df = df.copy()
    rho = df["ldaps_surface_0_sp"] / (287.05 * df["ldaps_heightAboveGround_2_t"])
    df["phy_density"] = rho
    df["phy_ws10_cu"] = df["ldaps_ws_10m"] ** 3
    df["phy_ws100_cu"] = df["gfs_ws_100m"] ** 3
    df["phy_rho_ws100_cu"] = rho * df["phy_ws100_cu"]
    df["phy_pc100"] = np.clip(df["gfs_ws_100m"], 0, 12) ** 3 / 12 ** 3
    ws50max = np.hypot(df["ldaps_heightAboveGround_50_50MUmax"], df["ldaps_heightAboveGround_50_50MVmax"])
    ws50min = np.hypot(df["ldaps_heightAboveGround_50_50MUmin"], df["ldaps_heightAboveGround_50_50MVmin"])
    df["phy_ws50max"] = ws50max
    df["phy_gust_range"] = ws50max - ws50min
    df["phy_gust_over_ws"] = ws50max / (df["ldaps_ws_10m"] + 0.5)
    df["phy_gfs_gust_factor"] = df["gfs_surface_0_gust"] / (df["gfs_ws_10m"] + 0.5)
    ws850 = np.hypot(df["gfs_isobaricInhPa_850_u"], df["gfs_isobaricInhPa_850_v"])
    df["phy_ws850"] = ws850
    df["phy_shear_850_100"] = ws850 - df["gfs_ws_100m"]
    cols = [c for c in df.columns if c.startswith("phy_")]
    return df, cols

def add_temporal(df):
    df = df.sort_values(["kpx_group", "forecast_kst_dtm"]).copy()
    for src in ["ldaps_ws_10m", "gfs_ws_100m"]:
        g = df.groupby("kpx_group")[src]
        p = "tmp_" + ("l10" if src.startswith("ldaps") else "g100")
        df[f"{p}_r3_mean"] = g.transform(lambda s: s.rolling(3, center=True, min_periods=1).mean())
        df[f"{p}_r3_std"] = g.transform(lambda s: s.rolling(3, center=True, min_periods=1).std())
        df[f"{p}_r7_mean"] = g.transform(lambda s: s.rolling(7, center=True, min_periods=1).mean())
        df[f"{p}_r7_std"] = g.transform(lambda s: s.rolling(7, center=True, min_periods=1).std())
        df[f"{p}_d1"] = g.transform(lambda s: s.diff(1))
        df[f"{p}_dm1"] = g.transform(lambda s: s.diff(-1))
        df[f"{p}_d3"] = g.transform(lambda s: s.diff(3))
        df[f"{p}_dm3"] = g.transform(lambda s: s.diff(-3))
    day_key = (pd.to_datetime(df["forecast_kst_dtm"]) - pd.Timedelta(hours=1)).dt.date
    day_grp = df.groupby([df["kpx_group"], day_key])["ldaps_ws_10m"]
    df["tmp_day_mean"] = day_grp.transform("mean")
    df["tmp_day_max"] = day_grp.transform("max")
    df["tmp_day_std"] = day_grp.transform("std")
    cols = [c for c in df.columns if c.startswith("tmp_")]
    return df, cols

def build_spatial_frames(info):
    centroids = info.groupby("kpx_group")[["lat", "lon"]].mean()
    out = {}
    for split, ld_path, gf_path in [("train", TRAIN_DIR / "ldaps_train.csv", TRAIN_DIR / "gfs_train.csv"),
                                    ("test", TEST_DIR / "ldaps_test.csv", TEST_DIR / "gfs_test.csv")]:
        ld = pd.read_csv(ld_path, encoding="utf-8-sig",
                         usecols=["forecast_kst_dtm", "grid_id", "latitude", "longitude",
                                  "heightAboveGround_10_10u", "heightAboveGround_10_10v"])
        ld["forecast_kst_dtm"] = pd.to_datetime(ld["forecast_kst_dtm"])
        ld["ws"] = np.hypot(ld["heightAboveGround_10_10u"], ld["heightAboveGround_10_10v"])
        grids = ld[["grid_id", "latitude", "longitude"]].drop_duplicates().set_index("grid_id")
        frames = []
        for group in [1, 2, 3]:
            d = haversine(centroids.loc[group, "lat"], centroids.loc[group, "lon"], grids["latitude"], grids["longitude"])
            near = d.nsmallest(4).index
            sub = ld[ld["grid_id"].isin(near)].groupby("forecast_kst_dtm")["ws"]
            agg = pd.DataFrame({"spa_l10_mean": sub.mean(), "spa_l10_std": sub.std(),
                                "spa_l10_range": sub.max() - sub.min()}).reset_index()
            agg["kpx_group"] = group
            frames.append(agg)
        ldaps_spa = pd.concat(frames, ignore_index=True)
        gf = pd.read_csv(gf_path, encoding="utf-8-sig",
                         usecols=["forecast_kst_dtm", "grid_id", "heightAboveGround_100_100u", "heightAboveGround_100_100v"])
        gf["forecast_kst_dtm"] = pd.to_datetime(gf["forecast_kst_dtm"])
        gf["ws100"] = np.hypot(gf["heightAboveGround_100_100u"], gf["heightAboveGround_100_100v"])
        sub = gf.groupby("forecast_kst_dtm")["ws100"]
        gfs_spa = pd.DataFrame({"spa_g100_mean": sub.mean(), "spa_g100_std": sub.std()}).reset_index()
        out[split] = (ldaps_spa, gfs_spa)
    return out

def build_v11(info):
    train_df, test_df, weather_cols_v10, sample_submission, _ = build_datasets()
    spa = build_spatial_frames(info)
    dfs = {}
    for split, df in [("train", train_df), ("test", test_df)]:
        df, phy_cols = add_physics(df)
        df, tmp_cols = add_temporal(df)
        ldaps_spa, gfs_spa = spa[split]
        df = df.merge(ldaps_spa, on=["forecast_kst_dtm", "kpx_group"], how="left")
        df = df.merge(gfs_spa, on="forecast_kst_dtm", how="left")
        dfs[split] = df
    spa_cols = ["spa_l10_mean", "spa_l10_std", "spa_l10_range", "spa_g100_mean", "spa_g100_std"]
    v10tps_cols = weather_cols_v10 + tmp_cols + phy_cols + spa_cols
    return dfs["train"], dfs["test"], v10tps_cols, sample_submission

In [5]:
# ═══════════════════════════════════════════════════════
# 4. spd 다격자 편차 피처
# ═══════════════════════════════════════════════════════
SPD_PATH = CACHE_DIR / "spatial_spd.parquet"
LD_VARS = {
    "ws10": lambda d: np.hypot(d["heightAboveGround_10_10u"], d["heightAboveGround_10_10v"]),
    "u10": lambda d: d["heightAboveGround_10_10u"], "v10": lambda d: d["heightAboveGround_10_10v"],
    "ws50x": lambda d: np.hypot(d["heightAboveGround_50_50MUmax"], d["heightAboveGround_50_50MVmax"]),
    "wsbl": lambda d: np.hypot(d["heightAboveGround_5_XBLWS"], d["heightAboveGround_5_YBLWS"]),
    "t2": lambda d: d["heightAboveGround_2_t"], "sp": lambda d: d["surface_0_sp"], "blh": lambda d: d["etc_0_blh"],
}
LD_SRC = ["heightAboveGround_10_10u", "heightAboveGround_10_10v", "heightAboveGround_50_50MUmax",
          "heightAboveGround_50_50MVmax", "heightAboveGround_5_XBLWS", "heightAboveGround_5_YBLWS",
          "heightAboveGround_2_t", "surface_0_sp", "etc_0_blh"]
GF_VARS = {
    "ws10": lambda d: np.hypot(d["heightAboveGround_10_10u"], d["heightAboveGround_10_10v"]),
    "ws100": lambda d: np.hypot(d["heightAboveGround_100_100u"], d["heightAboveGround_100_100v"]),
    "u100": lambda d: d["heightAboveGround_100_100u"], "v100": lambda d: d["heightAboveGround_100_100v"],
    "gust": lambda d: d["surface_0_gust"],
    "ws850": lambda d: np.hypot(d["isobaricInhPa_850_u"], d["isobaricInhPa_850_v"]),
    "t850": lambda d: d["isobaricInhPa_850_t"], "prmsl": lambda d: d["meanSea_0_prmsl"],
}
GF_SRC = ["heightAboveGround_10_10u", "heightAboveGround_10_10v", "heightAboveGround_100_100u",
          "heightAboveGround_100_100v", "surface_0_gust", "isobaricInhPa_850_u",
          "isobaricInhPa_850_v", "isobaricInhPa_850_t", "meanSea_0_prmsl"]

def _load(paths, src_cols):
    frames = []
    for p in paths:
        d = pd.read_csv(p, encoding="utf-8-sig", usecols=["forecast_kst_dtm", "grid_id", "latitude", "longitude", *src_cols])
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    return df

def _pivot(df, fn, master_idx, gid):
    v = df[["forecast_kst_dtm", "grid_id"]].copy()
    v["val"] = np.asarray(fn(df), dtype=np.float64)
    M = v.pivot_table(index="forecast_kst_dtm", columns="grid_id", values="val", aggfunc="mean")
    M = M.reindex(index=master_idx, columns=gid)
    row_mean = M.mean(axis=1)
    return M.apply(lambda c: c.fillna(row_mean))

def build_spatial(prefix, paths, src_cols, var_defs):
    df = _load(paths, src_cols)
    grids = df[["grid_id", "latitude", "longitude"]].drop_duplicates().sort_values("grid_id")
    gid = grids["grid_id"].to_numpy()
    master_idx = pd.Index(np.sort(df["forecast_kst_dtm"].unique()), name="forecast_kst_dtm")
    spd_cols = {}
    for name, fn in var_defs.items():
        M = _pivot(df, fn, master_idx, gid)
        V = M.to_numpy()
        mean = V.mean(axis=1)
        spd_cols[f"sd{prefix}_{name}_mean"] = mean
        for j, g in enumerate(gid):
            spd_cols[f"sd{prefix}_{name}_g{int(g)}"] = V[:, j] - mean
    return pd.DataFrame(spd_cols, index=master_idx).reset_index()

def build_spd(force=False):
    if not force and SPD_PATH.exists():
        spd = pd.read_parquet(SPD_PATH)
    else:
        t0 = time.time()
        ld_spd = build_spatial("L", [TRAIN_DIR / "ldaps_train.csv", TEST_DIR / "ldaps_test.csv"], LD_SRC, LD_VARS)
        gf_spd = build_spatial("G", [TRAIN_DIR / "gfs_train.csv", TEST_DIR / "gfs_test.csv"], GF_SRC, GF_VARS)
        spd = ld_spd.merge(gf_spd, on="forecast_kst_dtm", how="outer")
        spd.to_parquet(SPD_PATH, index=False)
        print(f"spd 캐시 생성 완료 ({time.time()-t0:.0f}s)")
    spd_cols = [c for c in spd.columns if c != "forecast_kst_dtm"]
    return spd, spd_cols

In [6]:
# ═══════════════════════════════════════════════════════
# 5. JMA 외부예보 수집
# ═══════════════════════════════════════════════════════
import requests

JMA_PATH = DATA_DIR / "external"
JMA_PATH.mkdir(parents=True, exist_ok=True)
BASE_URL = "https://previous-runs-api.open-meteo.com/v1/forecast"
MODELS = {"jma_gsm": ["2022", "2023", "2024", "2025"], "jma_msm": ["2023", "2024", "2025"]}
HOURLY_VARS = ["wind_speed_10m", "wind_direction_10m", "temperature_2m", "surface_pressure"]
LEADS = ["previous_day2", "previous_day1"]

def fetch_json(url, retries=3):
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=120)
            d = r.json()
            if "hourly" in d:
                return d
            if "reason" in d:
                raise RuntimeError(f"API error: {d['reason']}")
        except Exception as e:
            print(f"  재시도 {attempt+1}: {e}")
        time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"fetch 실패: {url[:120]}")

def get_centroids(info):
    return info.groupby("kpx_group")[["lat", "lon"]].mean()

def fetch_jma_prevruns(info, force=False):
    out_path = JMA_PATH / "jma_prevruns.csv"
    if not force and out_path.exists():
        return pd.read_csv(out_path, encoding="utf-8-sig", parse_dates=["forecast_kst_dtm"])
    centroids = get_centroids(info)
    hourly_param = ",".join(f"{v}_{lead}" for v in HOURLY_VARS for lead in LEADS)
    frames = []
    for group in [1, 2, 3]:
        lat = round(float(centroids.loc[group, "lat"]), 4)
        lon = round(float(centroids.loc[group, "lon"]), 4)
        for model, years in MODELS.items():
            for year in years:
                url = (f"{BASE_URL}?latitude={lat}&longitude={lon}&models={model}"
                       f"&hourly={hourly_param}&wind_speed_unit=ms"
                       f"&start_date={year}-01-01&end_date={year}-12-31&timezone=Asia%2FSeoul")
                d = fetch_json(url)
                h = d["hourly"]
                df = pd.DataFrame({"forecast_kst_dtm": pd.to_datetime(h["time"])})
                for v in HOURLY_VARS:
                    for lead in LEADS:
                        short = {"previous_day2": "d2", "previous_day1": "d1"}[lead]
                        df[f"{v}_{short}"] = h.get(f"{v}_{lead}")
                df["kpx_group"] = group
                df["model"] = model
                frames.append(df)
                time.sleep(1)
    out = pd.concat(frames, ignore_index=True)
    out.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {out_path} ({len(out)}행)")
    return out

def build_jma_frame(jma_df):
    frames = []
    for model, tag in [("jma_gsm", "jgsm"), ("jma_msm", "jmsm")]:
        sub = jma_df[jma_df["model"] == model].copy()
        wd_rad = np.deg2rad(sub["wind_direction_10m_d2"])
        out = pd.DataFrame({
            "forecast_kst_dtm": sub["forecast_kst_dtm"], "kpx_group": sub["kpx_group"],
            f"{tag}_ws": sub["wind_speed_10m_d2"], f"{tag}_wdsin": np.sin(wd_rad), f"{tag}_wdcos": np.cos(wd_rad),
        })
        if model == "jma_gsm":
            out[f"{tag}_t2m"] = sub["temperature_2m_d2"]
            out[f"{tag}_sp"] = sub["surface_pressure_d2"]
        frames.append(out)
    return frames[0].merge(frames[1], on=["forecast_kst_dtm", "kpx_group"], how="outer")

JG_COLS = ["jgsm_ws", "jgsm_wdsin", "jgsm_wdcos", "jgsm_t2m", "jgsm_sp", "jgsm_dis"]
JM_COLS = ["jmsm_ws", "jmsm_wdsin", "jmsm_wdcos", "jmsm_dis"]

In [7]:
# ═══════════════════════════════════════════════════════
# 6. DL — MTMLP + q60 비대칭 손실
# ═══════════════════════════════════════════════════════
GIDX = {1: 0, 2: 1, 3: 2}

class MTMLP(nn.Module):
    def __init__(self, n_feat, width=384, depth=3, p_drop=0.15, emb=8):
        super().__init__()
        self.emb = nn.Embedding(3, emb)
        layers, d_in = [], n_feat + emb
        for i in range(depth):
            layers += [nn.Linear(d_in, width), nn.BatchNorm1d(width), nn.SiLU(), nn.Dropout(p_drop)]
            d_in = width
        self.trunk = nn.Sequential(*layers)
        self.heads = nn.ModuleList([nn.Linear(width, 1) for _ in range(3)])

    def forward(self, x, g):
        h = self.trunk(torch.cat([x, self.emb(g)], dim=1))
        out = torch.zeros(x.shape[0], device=x.device)
        for gi in range(3):
            m = g == gi
            if m.any(): out[m] = self.heads[gi](h[m]).squeeze(1)
        return out

DELTA, TAU = 0.05, 0.60
def _huber(e):
    a = torch.abs(e)
    return torch.where(a <= DELTA, 0.5*e*e, DELTA*(a - 0.5*DELTA))

def q60_loss(pred, tgt, w):
    e = pred - tgt
    asym = torch.where(e < 0, 2.0*TAU, 2.0*(1.0-TAU))
    return (_huber(e) * asym * w).sum() / w.sum()

DL_CFG = dict(width=384, depth=3, p_drop=0.15, epochs=120, lr=1e-3, wd=1e-4)

def make_xy(df, cols, imputer, scaler, fit=False):
    sub = df.loc[df["target"].notna()]
    caps = sub["kpx_group"].map(lambda g: CAPACITY_KWH[f"kpx_group_{int(g)}"]).to_numpy()
    Xraw = sub[cols].to_numpy(dtype=np.float32)
    Xs = scaler.fit_transform(imputer.fit_transform(Xraw)) if fit else scaler.transform(imputer.transform(Xraw))
    g = sub["kpx_group"].map(GIDX).to_numpy().astype(np.int64)
    cf = (sub["target"].to_numpy() / caps).astype(np.float32)
    w = np.where(cf < 0.10, 0.05, 1.0).astype(np.float32)
    return Xs.astype(np.float32), g, cf, w

def train_model(Xtr, gtr, ytr, wtr, width, depth, p_drop, lr, wd, epochs, bs, seed, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    model = MTMLP(Xtr.shape[1], width, depth, p_drop).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xt = torch.tensor(Xtr, device=DEVICE); gt = torch.tensor(gtr, device=DEVICE)
    yt = torch.tensor(ytr, device=DEVICE); wt = torch.tensor(wtr, device=DEVICE)
    n = Xt.shape[0]
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            opt.zero_grad()
            loss = q60_loss(model(Xt[idx], gt[idx]), yt[idx], wt[idx])
            loss.backward(); opt.step()
        sched.step()
    return model

@torch.no_grad()
def predict(model, X, g, bs=8192):
    model.eval(); out = []
    Xt = torch.tensor(X, device=DEVICE); gt = torch.tensor(g, device=DEVICE)
    for i in range(0, Xt.shape[0], bs):
        out.append(model(Xt[i:i+bs], gt[i:i+bs]).cpu().numpy())
    return np.concatenate(out)

def dl_fold_predict(tr_df, va_df, cols, seeds, cfg=None):
    cfg = cfg or DL_CFG
    imp = SimpleImputer(strategy="median", keep_empty_features=True)
    sca = StandardScaler()
    Xtr, gtr, ytr, wtr = make_xy(tr_df, cols, imp, sca, fit=True)
    Xva = sca.transform(imp.transform(va_df[cols].to_numpy(dtype=np.float32))).astype(np.float32)
    gva = va_df["kpx_group"].map(GIDX).to_numpy().astype(np.int64)
    preds = []
    for s in range(seeds):
        m = train_model(Xtr, gtr, ytr, wtr, width=cfg["width"], depth=cfg["depth"],
                        p_drop=cfg["p_drop"], lr=cfg["lr"], wd=cfg["wd"],
                        epochs=cfg["epochs"], bs=1024, seed=s, verbose=False)
        preds.append(predict(m, Xva, gva))
        del m; torch.cuda.empty_cache()
    return np.mean(preds, axis=0)

In [8]:
# ═══════════════════════════════════════════════════════
# DL v2 — FiLM + 잔차 연결 (기존 MTMLP와 구조적으로 다른 아키텍처)
# ═══════════════════════════════════════════════════════
class FiLMBlock(nn.Module):
    """그룹 임베딩으로 이 블록의 출력을 그룹별로 scale·shift"""
    def __init__(self, width, emb_dim):
        super().__init__()
        self.fc = nn.Linear(width, width)
        self.bn = nn.BatchNorm1d(width)
        self.act = nn.SiLU()
        self.drop = nn.Dropout(0.15)
        self.film = nn.Linear(emb_dim, width * 2)  # gamma, beta 생성

    def forward(self, x, g_emb):
        h = self.drop(self.act(self.bn(self.fc(x))))
        gamma, beta = self.film(g_emb).chunk(2, dim=1)
        h = h * (1 + gamma) + beta          # FiLM 변조
        return x + h                         # 잔차 연결


class MTMLP_FiLM(nn.Module):
    def __init__(self, n_feat, width=384, depth=3, emb=8):
        super().__init__()
        self.emb = nn.Embedding(3, emb)
        self.input_proj = nn.Linear(n_feat, width)
        self.blocks = nn.ModuleList([FiLMBlock(width, emb) for _ in range(depth)])
        self.heads = nn.ModuleList([nn.Linear(width, 1) for _ in range(3)])

    def forward(self, x, g):
        g_emb = self.emb(g)
        h = self.input_proj(x)
        for block in self.blocks:
            h = block(h, g_emb)   # 매 층마다 그룹 정보 재주입
        out = torch.zeros(x.shape[0], device=x.device)
        for gi in range(3):
            m = g == gi
            if m.any(): out[m] = self.heads[gi](h[m]).squeeze(1)
        return out


def train_model_film(Xtr, gtr, ytr, wtr, width, depth, lr, wd, epochs, bs, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = MTMLP_FiLM(Xtr.shape[1], width, depth).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xt = torch.tensor(Xtr, device=DEVICE); gt = torch.tensor(gtr, device=DEVICE)
    yt = torch.tensor(ytr, device=DEVICE); wt = torch.tensor(wtr, device=DEVICE)
    n = Xt.shape[0]
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            opt.zero_grad()
            loss = q60_loss(model(Xt[idx], gt[idx]), yt[idx], wt[idx])  # 손실함수는 기존 q60 그대로
            loss.backward(); opt.step()
        sched.step()
    return model


def dl_fold_predict_film(tr_df, va_df, cols, seeds, cfg=None):
    cfg = cfg or DL_CFG
    imp = SimpleImputer(strategy="median", keep_empty_features=True)
    sca = StandardScaler()
    Xtr, gtr, ytr, wtr = make_xy(tr_df, cols, imp, sca, fit=True)
    Xva = sca.transform(imp.transform(va_df[cols].to_numpy(dtype=np.float32))).astype(np.float32)
    gva = va_df["kpx_group"].map(GIDX).to_numpy().astype(np.int64)
    preds = []
    for s in range(seeds):
        m = train_model_film(Xtr, gtr, ytr, wtr, width=cfg["width"], depth=cfg["depth"],
                             lr=cfg["lr"], wd=cfg["wd"], epochs=cfg["epochs"], bs=1024, seed=s)
        preds.append(predict(m, Xva, gva))
        del m; torch.cuda.empty_cache()
    return np.mean(preds, axis=0)

In [9]:
# ═══════════════════════════════════════════════════════
# 7. 후처리 (단순화: 편향보정 + rolling + clip, 스냅 없음)
# ═══════════════════════════════════════════════════════
def postprocess_simple(cf, cap, ws, group, bias_map, clip_lo=0.16, alpha=0.40):
    base = apply_bias_correction_v2(cf * cap, ws, group, bias_map)
    base = apply_rolling_smoothing(pd.Series(base), 3, alpha).to_numpy()
    return np.clip(base, cap * clip_lo, cap)

def score_from_tidy(tidy, pred_col):
    ans, prd = [], []
    for fname, sub in tidy.groupby("fold", sort=False):
        idx = np.sort(sub["forecast_kst_dtm"].unique())
        out = []
        for val in ("target", pred_col):
            m = sub.pivot_table(index="forecast_kst_dtm", columns="group", values=val)
            m.columns = [f"kpx_group_{int(c)}" for c in m.columns]
            out.append(m.reindex(index=idx, columns=TARGET_COLS).fillna(0.0))
        ans.append(out[0]); prd.append(out[1])
    return metric(pd.concat(ans, ignore_index=True), pd.concat(prd, ignore_index=True))

def build_slices(cache):
    return [((fn, int(g)), sub.sort_values("forecast_kst_dtm")) for (fn, g), sub in cache.groupby(["fold", "group"], sort=False)]

In [10]:
# ═══════════════════════════════════════════════════════
# 8. SCADA 가용성
# ═══════════════════════════════════════════════════════
GROUP_TURBINES = {
    1: [("vestas", i) for i in range(1, 7)],
    2: [("vestas", i) for i in range(7, 13)],
    3: [("unison", i) for i in range(1, 6)],
}

def load_scada():
    v = pd.read_csv(TRAIN_DIR / "scada_vestas_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])
    u = pd.read_csv(TRAIN_DIR / "scada_unison_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])
    return v.merge(u, on="kst_dtm", how="outer").sort_values("kst_dtm").reset_index(drop=True)

def compute_availability(sc):
    sc = sc.copy()
    sc["hour"] = sc["kst_dtm"].dt.floor("h")
    rows = []
    for g, turbines in GROUP_TURBINES.items():
        p_cols = [f"{mk}_wtg{i:02d}_power_kw10m" for mk, i in turbines]
        w_cols = [f"{mk}_wtg{i:02d}_ws" for mk, i in turbines]
        grp = sc[["hour"] + p_cols + w_cols]
        med_ws = grp[w_cols].median(axis=1)
        agg = {}
        for (mk, i), pc, wc in zip(turbines, p_cols, w_cols):
            ws_eff = grp[wc].fillna(med_ws)
            windy = ws_eff >= 5.0
            prod = grp[pc] > 1.0
            dead_strict = (~prod.fillna(False)) & windy
            slots = grp.groupby("hour")[pc].size()
            agg[f"ds_{mk}{i}"] = dead_strict.groupby(grp["hour"]).sum()
            agg[f"n_{mk}{i}"] = slots
        A = pd.DataFrame(agg)
        n_t = len(turbines)
        un_s = sum((A[f"ds_{mk}{i}"] >= np.maximum(A[f"n_{mk}{i}"] - 1, 5)).astype(int) for mk, i in turbines)
        rows.append(pd.DataFrame({"forecast_kst_dtm": A.index, "group": g, "avail_strict": 1 - un_s / n_t}))
    return pd.concat(rows, ignore_index=True)

def avail_factor(av_vals, low=0.05):
    a = np.nan_to_num(av_vals.astype(float), nan=1.0)
    return np.where(a < 0.9, low, np.where(a < 0.999, 0.3, 1.0))

def make_sample_weight_scada(y, capacity, av_series, low_weight=0.05):
    base_w = make_sample_weight(y, capacity, low_weight)
    af = avail_factor(av_series.to_numpy())
    return base_w * af

In [11]:
# ═══════════════════════════════════════════════════════
# GBM 캐시 생성 (4-fold Q1~Q4 + 홀드아웃, 총 5개 fold 순회)
# ═══════════════════════════════════════════════════════
RATED_CF = 0.90
N_CLF_SEEDS = 3

def build_pooled_matrices(tr_pool, va_g, weather_cols):
    tr_feat = pd.concat([calendar_features(tr_pool["forecast_kst_dtm"]).reset_index(drop=True),
                         tr_pool[weather_cols].reset_index(drop=True)], axis=1)
    va_feat = pd.concat([calendar_features(va_g["forecast_kst_dtm"]).reset_index(drop=True),
                         va_g[weather_cols].reset_index(drop=True)], axis=1)
    imputer = SimpleImputer(strategy="median")
    X_tr = pd.DataFrame(imputer.fit_transform(tr_feat), columns=tr_feat.columns)
    X_va = pd.DataFrame(imputer.transform(va_feat), columns=va_feat.columns)
    return X_tr, X_va

def cache_gbm(train_df, cols, jobs=2, gbm_seeds=3):
    rows, bias_maps = [], {}
    for fname, fold in tqdm(FOLD_LIST, desc="GBM fold 학습"):  # ← Q1,Q2,Q3,Q4,hold 총 5개
        tf = time.time()
        tr_all = train_df.loc[train_df["forecast_kst_dtm"] <= fold["train_end"]]
        va_all = train_df.loc[(train_df["forecast_kst_dtm"] >= fold["valid_start"]) & (train_df["forecast_kst_dtm"] <= fold["valid_end"])]
        tr_pool = tr_all.loc[tr_all["target"].notna()]

        bias_data = estimate_bias_data(train_df, cols, fold["train_end"], 500, jobs)
        bias_maps[fname] = bias_map_from_data(bias_data, {g: COARSE_BINS for g in bias_data})

        clf = []
        for group in [1, 2, 3]:
            cap = CAPACITY_KWH[f"kpx_group_{group}"]
            tr_g = tr_all.loc[(tr_all["kpx_group"] == group) & tr_all["target"].notna()].copy()
            va_g = va_all.loc[va_all["kpx_group"] == group].sort_values("forecast_kst_dtm").copy()
            if len(tr_g) == 0 or len(va_g) == 0: continue

            X_tr, X_va = prepare_matrices(tr_g, va_g, cols)
            y_tr = tr_g["target"]; w_tr = make_sample_weight(y_tr.to_numpy(), cap)
            pl = np.mean([lgb.LGBMRegressor(**lgbm_params(500, jobs, s)).fit(X_tr, y_tr, sample_weight=w_tr).predict(X_va) for s in range(gbm_seeds)], axis=0)
            px = np.mean([xgbm.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.03, min_child_weight=5,
                          subsample=0.8, colsample_bytree=0.8, random_state=s, n_jobs=jobs, verbosity=0,
                          objective="reg:absoluteerror").fit(X_tr, y_tr, sample_weight=w_tr).predict(X_va) for s in range(gbm_seeds)], axis=0)
            gbm_cf = (0.5 * pl + 0.5 * px) / cap

            X_pool_tr, X_pool_va = build_pooled_matrices(tr_pool, va_g, cols)
            if not clf:
                cp = tr_pool["kpx_group"].map(lambda g: CAPACITY_KWH[f"kpx_group_{int(g)}"]).to_numpy()
                yA = (tr_pool["target"].to_numpy() >= cp * RATED_CF).astype(int)
                for s in range(N_CLF_SEEDS):
                    c = lgb.LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.03, num_leaves=31,
                                           min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                                           random_state=s, n_jobs=jobs, verbose=-1)
                    c.fit(X_pool_tr, yA); clf.append(c)
            pA = np.mean([c.predict_proba(X_pool_va)[:, 1] for c in clf], axis=0)

            rows.append(pd.DataFrame({"fold": fname, "group": group,
                                      "forecast_kst_dtm": va_g["forecast_kst_dtm"].to_numpy(),
                                      "target": va_g["target"].to_numpy(), "ws": va_g["ldaps_ws_10m"].to_numpy(),
                                      "gbm_cf": gbm_cf, "pA": pA}))
        print(f"  [gbm-cache:{fname}] {time.time()-tf:.0f}s", flush=True)

    out = pd.concat(rows, ignore_index=True)
    return out, bias_maps


# ═══════════════════════════════════════════════════════
# DL 캐시 생성 (동일하게 5개 fold 순회)
# ═══════════════════════════════════════════════════════
def cache_dl(train_df, cols, seeds=3, cfg=None):
    cfg = cfg or DL_CFG
    dl_rows = []
    for fname, fold in tqdm(FOLD_LIST, desc="DL fold 학습"):  # ← 동일하게 5개
        tf = time.time()
        tr_all = train_df.loc[train_df["forecast_kst_dtm"] <= fold["train_end"]]
        va_all = train_df.loc[(train_df["forecast_kst_dtm"] >= fold["valid_start"]) & (train_df["forecast_kst_dtm"] <= fold["valid_end"])].sort_values("forecast_kst_dtm")
        tr_pool = tr_all.loc[tr_all["target"].notna()].copy()

        dl_pred = dl_fold_predict(tr_pool, va_all, cols, seeds=seeds, cfg=cfg)
        dl_rows.append(pd.DataFrame({"fold": fname, "group": va_all["kpx_group"].to_numpy(),
                                     "forecast_kst_dtm": va_all["forecast_kst_dtm"].to_numpy(), "dl_q60": dl_pred}))
        print(f"  [dl-cache:{fname}] {time.time()-tf:.0f}s", flush=True)
    return pd.concat(dl_rows, ignore_index=True)

In [12]:
# ── 데이터 준비 ──
train_df_tps, test_df_tps, cols_tps, sample_submission = build_v11(build_datasets()[4])
# 주의: info를 따로 받으려면 build_datasets()를 한 번 더 호출하는 게 안전합니다.
_, _, _, sample_submission, info = build_datasets()
train_df_tps, test_df_tps, cols_tps, _ = build_v11(info)

spd, spd_cols = build_spd()
train_df_tps = train_df_tps.merge(spd, on="forecast_kst_dtm", how="left")
test_df_tps = test_df_tps.merge(spd, on="forecast_kst_dtm", how="left")

jma_raw = fetch_jma_prevruns(info)
jma = build_jma_frame(jma_raw)
train_df_tps = train_df_tps.merge(jma, on=["forecast_kst_dtm", "kpx_group"], how="left")
test_df_tps = test_df_tps.merge(jma, on=["forecast_kst_dtm", "kpx_group"], how="left")
for df in [train_df_tps, test_df_tps]:
    df["jgsm_dis"] = (df["jgsm_ws"] - df["ldaps_ws_10m"]).abs()
    df["jmsm_dis"] = (df["jmsm_ws"] - df["ldaps_ws_10m"]).abs()

cols_gspd = cols_tps + JG_COLS + JM_COLS + spd_cols
print("최종 피처 수:", len(cols_gspd))  # 348 나와야 함

sc = load_scada()
av = compute_availability(sc)
av["forecast_kst_dtm"] = pd.to_datetime(av["forecast_kst_dtm"])

# ── GBM/DL 캐시 (이미 만들어둔 게 있으면 재사용) ──
GBM_CACHE_PATH = CACHE_DIR / "gbm_gspd_cache.parquet"
BIAS_CACHE_PATH = CACHE_DIR / "gbm_gspd_bias.pkl"
DL_CACHE_PATH = CACHE_DIR / "dl_q60_cache.parquet"
DL_FILM_CACHE_PATH = CACHE_DIR / "dl_film_cache.parquet"

if DL_FILM_CACHE_PATH.exists():
    dl_film_cache = pd.read_parquet(DL_FILM_CACHE_PATH)
    print("FiLM DL 캐시 로드:", dl_film_cache.shape)
else:
    dl_film_rows = []
    for fname, fold in tqdm(FOLD_LIST, desc="FiLM DL fold 학습"):
        tf = time.time()
        tr_all = train_df_tps.loc[train_df_tps["forecast_kst_dtm"] <= fold["train_end"]]
        va_all = train_df_tps.loc[(train_df_tps["forecast_kst_dtm"] >= fold["valid_start"]) &
                                   (train_df_tps["forecast_kst_dtm"] <= fold["valid_end"])].sort_values("forecast_kst_dtm")
        tr_pool = tr_all.loc[tr_all["target"].notna()].copy()
        dl_pred = dl_fold_predict_film(tr_pool, va_all, cols_gspd, seeds=3, cfg=DL_CFG)
        dl_film_rows.append(pd.DataFrame({"fold": fname, "group": va_all["kpx_group"].to_numpy(),
                                          "forecast_kst_dtm": va_all["forecast_kst_dtm"].to_numpy(), "dl_film": dl_pred}))
        print(f"  [film-cache:{fname}] {time.time()-tf:.0f}s", flush=True)
    dl_film_cache = pd.concat(dl_film_rows, ignore_index=True)
    dl_film_cache.to_parquet(DL_FILM_CACHE_PATH, index=False)

gbm_cache = pd.read_parquet(GBM_CACHE_PATH)
with open(BIAS_CACHE_PATH, "rb") as f:
    bias_maps = pickle.load(f)
dl_cache = pd.read_parquet(DL_CACHE_PATH)

cache = gbm_cache.merge(dl_cache, on=["fold", "group", "forecast_kst_dtm"], how="left")
assert cache["dl_q60"].notna().all()

# ── 검증 (w=0.65 고정, 스냅 없음) ──
def score_simplified(cache, bias_maps, w=0.65):
    slices = build_slices(cache)
    parts = []
    for (fname, group), sub in slices:
        cap = CAPACITY_KWH[f"kpx_group_{group}"]
        cf = w * sub["gbm_cf"].to_numpy() + (1 - w) * sub["dl_q60"].to_numpy()
        pred = postprocess_simple(cf, cap, sub["ws"].to_numpy(), group, bias_maps[fname])
        parts.append(sub[["fold", "group", "forecast_kst_dtm", "target"]].assign(pred=pred))
    tidy = pd.concat(parts, ignore_index=True)
    hold = "hold_2024full"
    return score_from_tidy(tidy[tidy["fold"] != hold], "pred"), score_from_tidy(tidy[tidy["fold"] == hold], "pred")

cv_score, hold_score = score_simplified(cache, bias_maps, w=0.65)
print(f"CV Score={cv_score[0]:.4f} | HOLD Score={hold_score[0]:.4f}")

/tmp/ipykernel_5568/3506793104.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  info["kpx_group"] = info["kpx_group"].ffill().astype(int)


v10 피처 개수: 87


/tmp/ipykernel_5568/3506793104.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  info["kpx_group"] = info["kpx_group"].ffill().astype(int)


v10 피처 개수: 87


/tmp/ipykernel_5568/3506793104.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  info["kpx_group"] = info["kpx_group"].ffill().astype(int)


v10 피처 개수: 87


/tmp/ipykernel_5568/3506793104.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  info["kpx_group"] = info["kpx_group"].ffill().astype(int)


v10 피처 개수: 87
최종 피처 수: 348
FiLM DL 캐시 로드: (52704, 4)
CV Score=0.6387 | HOLD Score=0.6366


In [13]:
KEY = ["fold", "group", "forecast_kst_dtm"]
cache_v2 = gbm_cache.merge(dl_cache, on=KEY, how="left").merge(dl_film_cache, on=KEY, how="left")

def score_dl_variant(cache, bias_maps, w, dl_col):
    slices = build_slices(cache)
    parts = []
    for (fname, group), sub in slices:
        cap = CAPACITY_KWH[f"kpx_group_{group}"]
        cf = w * sub["gbm_cf"].to_numpy() + (1 - w) * sub[dl_col].to_numpy()
        pred = postprocess_simple(cf, cap, sub["ws"].to_numpy(), group, bias_maps[fname])
        parts.append(sub[["fold", "group", "forecast_kst_dtm", "target"]].assign(pred=pred))
    tidy = pd.concat(parts, ignore_index=True)
    hold = "hold_2024full"
    return score_from_tidy(tidy[tidy["fold"] != hold], "pred"), score_from_tidy(tidy[tidy["fold"] == hold], "pred")

print("### 기존 DL(q60/MTMLP) ###")
cv1, hd1 = score_dl_variant(cache_v2, bias_maps, w=0.65, dl_col="dl_q60")
print(f"CV={cv1[0]:.4f} | HOLD={hd1[0]:.4f}")

print("\n### FiLM DL 단독 ###")
cv2, hd2 = score_dl_variant(cache_v2, bias_maps, w=0.65, dl_col="dl_film")
print(f"CV={cv2[0]:.4f} | HOLD={hd2[0]:.4f}")

# DL끼리 상관관계 확인 — 진짜 다양성이 생겼는지
corr = cache_v2[["dl_q60", "dl_film"]].corr().iloc[0, 1]
print(f"\ncorr(dl_q60, dl_film) = {corr:.4f}  (0.998 미만이면 유의미한 다양성)")

# 두 DL을 평균낸 버전도 확인
cache_v2["dl_avg"] = 0.5 * cache_v2["dl_q60"] + 0.5 * cache_v2["dl_film"]
print("\n### DL 앙상블(q60+film 평균) ###")
cv3, hd3 = score_dl_variant(cache_v2, bias_maps, w=0.65, dl_col="dl_avg")
print(f"CV={cv3[0]:.4f} | HOLD={hd3[0]:.4f}")

### 기존 DL(q60/MTMLP) ###
CV=0.6387 | HOLD=0.6366

### FiLM DL 단독 ###
CV=0.6380 | HOLD=0.6359

corr(dl_q60, dl_film) = 0.9859  (0.998 미만이면 유의미한 다양성)

### DL 앙상블(q60+film 평균) ###
CV=0.6387 | HOLD=0.6362


In [14]:
# ═══════════════════════════════════════════════════════
# FiLM DL v2 — 입력 정규화 추가 + 깊이 조정 + 학습 설정 분리
# ═══════════════════════════════════════════════════════
class MTMLP_FiLM_v2(nn.Module):
    def __init__(self, n_feat, width=384, depth=4, emb=8, p_drop=0.15):
        super().__init__()
        self.emb = nn.Embedding(3, emb)
        # 입력 projection에도 정규화/활성화 추가 (잔차 시작점을 안정화)
        self.input_proj = nn.Sequential(
            nn.Linear(n_feat, width), nn.BatchNorm1d(width), nn.SiLU(), nn.Dropout(p_drop)
        )
        self.blocks = nn.ModuleList([FiLMBlock(width, emb) for _ in range(depth)])
        self.heads = nn.ModuleList([nn.Linear(width, 1) for _ in range(3)])

    def forward(self, x, g):
        g_emb = self.emb(g)
        h = self.input_proj(x)
        for block in self.blocks:
            h = block(h, g_emb)
        out = torch.zeros(x.shape[0], device=x.device)
        for gi in range(3):
            m = g == gi
            if m.any(): out[m] = self.heads[gi](h[m]).squeeze(1)
        return out


def train_model_film_v2(Xtr, gtr, ytr, wtr, width, depth, p_drop, lr, wd, epochs, bs, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = MTMLP_FiLM_v2(Xtr.shape[1], width, depth, p_drop=p_drop).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xt = torch.tensor(Xtr, device=DEVICE); gt = torch.tensor(gtr, device=DEVICE)
    yt = torch.tensor(ytr, device=DEVICE); wt = torch.tensor(wtr, device=DEVICE)
    n = Xt.shape[0]
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            opt.zero_grad()
            loss = q60_loss(model(Xt[idx], gt[idx]), yt[idx], wt[idx])
            loss.backward(); opt.step()
        sched.step()
    return model


def dl_fold_predict_film_v2(tr_df, va_df, cols, seeds, cfg):
    imp = SimpleImputer(strategy="median", keep_empty_features=True)
    sca = StandardScaler()
    Xtr, gtr, ytr, wtr = make_xy(tr_df, cols, imp, sca, fit=True)
    Xva = sca.transform(imp.transform(va_df[cols].to_numpy(dtype=np.float32))).astype(np.float32)
    gva = va_df["kpx_group"].map(GIDX).to_numpy().astype(np.int64)
    preds = []
    for s in range(seeds):
        m = train_model_film_v2(Xtr, gtr, ytr, wtr, width=cfg["width"], depth=cfg["depth"],
                                 p_drop=cfg["p_drop"], lr=cfg["lr"], wd=cfg["wd"],
                                 epochs=cfg["epochs"], bs=1024, seed=s)
        preds.append(predict(m, Xva, gva))
        del m; torch.cuda.empty_cache()
    return np.mean(preds, axis=0)

In [15]:
CFG_CANDIDATES = {
    "base":        dict(width=384, depth=4, p_drop=0.15, epochs=120, lr=1e-3,  wd=1e-4),
    "lower_lr":    dict(width=384, depth=4, p_drop=0.15, epochs=180, lr=5e-4,  wd=1e-4),
    "lower_lr2":   dict(width=384, depth=4, p_drop=0.15, epochs=200, lr=3e-4,  wd=1e-4),
    "deeper":      dict(width=384, depth=5, p_drop=0.20, epochs=150, lr=5e-4,  wd=1e-4),
}

fold_q4 = FOLDS_Q[-1]
tr_all = train_df_tps.loc[train_df_tps["forecast_kst_dtm"] <= fold_q4["train_end"]]
va_all = train_df_tps.loc[(train_df_tps["forecast_kst_dtm"] >= fold_q4["valid_start"]) &
                           (train_df_tps["forecast_kst_dtm"] <= fold_q4["valid_end"])].sort_values("forecast_kst_dtm")
tr_pool = tr_all.loc[tr_all["target"].notna()].copy()

# 같은 fold의 GBM 캐시(gbm_cf)를 가져와서 블렌드 스코어까지 바로 확인
gbm_q4 = gbm_cache[gbm_cache["fold"] == "fold_2024Q4"].sort_values(["group", "forecast_kst_dtm"])

for name, cfg in CFG_CANDIDATES.items():
    t0 = time.time()
    dl_pred = dl_fold_predict_film_v2(tr_pool, va_all, cols_gspd, seeds=2, cfg=cfg)  # 스캔 단계는 2-seed로 빠르게
    va_all_c = va_all.copy()
    va_all_c["dl_pred"] = dl_pred

    # 블렌드 스코어 계산 (w=0.65)
    merged = gbm_q4.merge(va_all_c[["forecast_kst_dtm", "kpx_group", "dl_pred"]].rename(columns={"kpx_group": "group"}),
                           on=["forecast_kst_dtm", "group"], how="left")
    parts = []
    for group in [1, 2, 3]:
        g_sub = merged[merged["group"] == group].sort_values("forecast_kst_dtm")
        cap = CAPACITY_KWH[f"kpx_group_{group}"]
        cf = 0.65 * g_sub["gbm_cf"].to_numpy() + 0.35 * g_sub["dl_pred"].to_numpy()
        pred = postprocess_simple(cf, cap, g_sub["ws"].to_numpy(), group, bias_maps["fold_2024Q4"])
        parts.append(g_sub[["group", "forecast_kst_dtm", "target"]].assign(pred=pred))
    tidy = pd.concat(parts, ignore_index=True)
    tidy["fold"] = "fold_2024Q4"
    s, n, f = score_from_tidy(tidy, "pred")
    print(f"{name}: Score={s:.4f} 1-NMAE={n:.4f} FiCR={f:.4f} ({time.time()-t0:.0f}s)")

KeyboardInterrupt: 

In [ ]:
CFG_ROUND2 = {
    "deeper_base":     dict(width=384, depth=5, p_drop=0.20, epochs=150, lr=5e-4, wd=1e-4),  # 재확인
    "deeper_lowerlr":  dict(width=384, depth=5, p_drop=0.20, epochs=180, lr=3e-4, wd=1e-4),
    "deeper_wider":    dict(width=512, depth=5, p_drop=0.20, epochs=150, lr=5e-4, wd=1e-4),
    "deeper_moredrop": dict(width=384, depth=5, p_drop=0.25, epochs=150, lr=5e-4, wd=1e-4),
}

for name, cfg in CFG_ROUND2.items():
    t0 = time.time()
    dl_pred = dl_fold_predict_film_v2(tr_pool, va_all, cols_gspd, seeds=2, cfg=cfg)
    va_all_c = va_all.copy()
    va_all_c["dl_pred"] = dl_pred
    merged = gbm_q4.merge(va_all_c[["forecast_kst_dtm", "kpx_group", "dl_pred"]].rename(columns={"kpx_group": "group"}),
                           on=["forecast_kst_dtm", "group"], how="left")
    parts = []
    for group in [1, 2, 3]:
        g_sub = merged[merged["group"] == group].sort_values("forecast_kst_dtm")
        cap = CAPACITY_KWH[f"kpx_group_{group}"]
        cf = 0.65 * g_sub["gbm_cf"].to_numpy() + 0.35 * g_sub["dl_pred"].to_numpy()
        pred = postprocess_simple(cf, cap, g_sub["ws"].to_numpy(), group, bias_maps["fold_2024Q4"])
        parts.append(g_sub[["group", "forecast_kst_dtm", "target"]].assign(pred=pred))
    tidy = pd.concat(parts, ignore_index=True)
    tidy["fold"] = "fold_2024Q4"
    s, n, f = score_from_tidy(tidy, "pred")
    print(f"{name}: Score={s:.4f} 1-NMAE={n:.4f} FiCR={f:.4f} ({time.time()-t0:.0f}s)")

In [16]:
FILM_V2_CFG = dict(width=512, depth=5, p_drop=0.20, epochs=150, lr=5e-4, wd=1e-4)
DL_FILM_V2_CACHE_PATH = CACHE_DIR / "dl_film_v2_cache.parquet"

if DL_FILM_V2_CACHE_PATH.exists():
    dl_film_v2_cache = pd.read_parquet(DL_FILM_V2_CACHE_PATH)
    print("FiLM v2 캐시 로드:", dl_film_v2_cache.shape)
else:
    dl_film_v2_rows = []
    for fname, fold in tqdm(FOLD_LIST, desc="FiLM v2 fold 학습"):
        tf = time.time()
        tr_all = train_df_tps.loc[train_df_tps["forecast_kst_dtm"] <= fold["train_end"]]
        va_all = train_df_tps.loc[(train_df_tps["forecast_kst_dtm"] >= fold["valid_start"]) &
                                   (train_df_tps["forecast_kst_dtm"] <= fold["valid_end"])].sort_values("forecast_kst_dtm")
        tr_pool = tr_all.loc[tr_all["target"].notna()].copy()
        dl_pred = dl_fold_predict_film_v2(tr_pool, va_all, cols_gspd, seeds=3, cfg=FILM_V2_CFG)  # 본 검증은 3-seed
        dl_film_v2_rows.append(pd.DataFrame({"fold": fname, "group": va_all["kpx_group"].to_numpy(),
                                             "forecast_kst_dtm": va_all["forecast_kst_dtm"].to_numpy(), "dl_film_v2": dl_pred}))
        print(f"  [filmv2-cache:{fname}] {time.time()-tf:.0f}s", flush=True)
    dl_film_v2_cache = pd.concat(dl_film_v2_rows, ignore_index=True)
    dl_film_v2_cache.to_parquet(DL_FILM_V2_CACHE_PATH, index=False)

# ── 비교 ──
KEY = ["fold", "group", "forecast_kst_dtm"]
cache_v3 = gbm_cache.merge(dl_cache, on=KEY, how="left").merge(dl_film_v2_cache, on=KEY, how="left")

def score_dl_variant2(cache, bias_maps, w, dl_col):
    slices = build_slices(cache)
    parts = []
    for (fname, group), sub in slices:
        cap = CAPACITY_KWH[f"kpx_group_{group}"]
        cf = w * sub["gbm_cf"].to_numpy() + (1 - w) * sub[dl_col].to_numpy()
        pred = postprocess_simple(cf, cap, sub["ws"].to_numpy(), group, bias_maps[fname])
        parts.append(sub[["fold", "group", "forecast_kst_dtm", "target"]].assign(pred=pred))
    tidy = pd.concat(parts, ignore_index=True)
    hold = "hold_2024full"
    cv = score_from_tidy(tidy[tidy["fold"] != hold], "pred")
    hd = score_from_tidy(tidy[tidy["fold"] == hold], "pred")
    fold_scores = tidy.groupby("fold").apply(lambda t: score_from_tidy(t, "pred")[0] if len(t) else np.nan)
    return cv, hd, fold_scores

print("### 기존 DL(q60) ###")
cv1, hd1, fs1 = score_dl_variant2(cache_v3, bias_maps, w=0.65, dl_col="dl_q60")
print(f"CV={cv1[0]:.4f} | HOLD={hd1[0]:.4f}")

print("\n### FiLM v2 (deeper_wider) ###")
cv2, hd2, fs2 = score_dl_variant2(cache_v3, bias_maps, w=0.65, dl_col="dl_film_v2")
print(f"CV={cv2[0]:.4f} | HOLD={hd2[0]:.4f}")

corr = cache_v3[["dl_q60", "dl_film_v2"]].corr().iloc[0, 1]
print(f"\ncorr(dl_q60, dl_film_v2) = {corr:.4f}")

cache_v3["dl_avg_v2"] = 0.5 * cache_v3["dl_q60"] + 0.5 * cache_v3["dl_film_v2"]
print("\n### DL 앙상블(q60+film_v2 평균) ###")
cv3s, hd3s, fs3 = score_dl_variant2(cache_v3, bias_maps, w=0.65, dl_col="dl_avg_v2")
print(f"CV={cv3s[0]:.4f} | HOLD={hd3s[0]:.4f}")

print("\nfold별 스코어 비교:")
print(pd.DataFrame({"기존": fs1, "film_v2": fs2, "앙상블": fs3}))

FiLM v2 캐시 로드: (52704, 4)
### 기존 DL(q60) ###


/tmp/ipykernel_5568/2767932077.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold_scores = tidy.groupby("fold").apply(lambda t: score_from_tidy(t, "pred")[0] if len(t) else np.nan)
/tmp/ipykernel_5568/2767932077.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold_scores = tidy.groupby("fold").apply(lambda t: score_from_tidy(t, "pred")[0] if len(t) else np.nan)


CV=0.6387 | HOLD=0.6366

### FiLM v2 (deeper_wider) ###
CV=0.6387 | HOLD=0.6356

corr(dl_q60, dl_film_v2) = 0.9909

### DL 앙상블(q60+film_v2 평균) ###
CV=0.6386 | HOLD=0.6360

fold별 스코어 비교:
                     기존   film_v2       앙상블
fold                                       
fold_2024Q1    0.619058  0.618011  0.618237
fold_2024Q2    0.647562  0.647143  0.647182
fold_2024Q3    0.649915  0.645717  0.648344
fold_2024Q4    0.643287  0.647627  0.645114
hold_2024full  0.636590  0.635610  0.635982


/tmp/ipykernel_5568/2767932077.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold_scores = tidy.groupby("fold").apply(lambda t: score_from_tidy(t, "pred")[0] if len(t) else np.nan)


In [17]:
W, CLIP_LO, ROLL_ALPHA = 0.65, 0.16, 0.40
final_bias_data = estimate_bias_data(train_df_tps, cols_gspd, train_df_tps["forecast_kst_dtm"].max(), 500, 2)
final_bias_map = bias_map_from_data(final_bias_data, {g: COARSE_BINS for g in final_bias_data})

predictions_final = pd.DataFrame(index=test_df_tps.index)
for group in tqdm([1, 2, 3], desc="최종 학습"):
    target_col = f"kpx_group_{group}"
    cap = CAPACITY_KWH[target_col]
    tr_g = train_df_tps.loc[(train_df_tps["kpx_group"] == group) & train_df_tps["target"].notna()].copy()
    te_g = test_df_tps.loc[test_df_tps["kpx_group"] == group].sort_values("forecast_kst_dtm").copy()
    tr_g = tr_g.merge(av[av["group"] == group][["forecast_kst_dtm", "avail_strict"]], on="forecast_kst_dtm", how="left")
    tr_g["avail_strict"] = tr_g["avail_strict"].fillna(1.0)

    X_tr, X_te = prepare_matrices(tr_g, te_g, cols_gspd)
    y_tr = tr_g["target"]
    w_tr = make_sample_weight_scada(y_tr.to_numpy(), cap, tr_g["avail_strict"])

    pl = np.mean([lgb.LGBMRegressor(**lgbm_params(500, 2, s)).fit(X_tr, y_tr, sample_weight=w_tr).predict(X_te) for s in range(3)], axis=0)
    px = np.mean([xgbm.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.03, min_child_weight=5,
                  subsample=0.8, colsample_bytree=0.8, random_state=s, n_jobs=2, verbosity=0,
                  objective="reg:absoluteerror").fit(X_tr, y_tr, sample_weight=w_tr).predict(X_te) for s in range(3)], axis=0)
    gbm_cf_te = (0.5 * pl + 0.5 * px) / cap
    dl_cf_te = dl_fold_predict(tr_g, te_g, cols_gspd, seeds=3, cfg=DL_CFG)

    cf_final = W * gbm_cf_te + (1 - W) * dl_cf_te
    pred = postprocess_simple(cf_final, cap, te_g["ldaps_ws_10m"].to_numpy(), group, final_bias_map, clip_lo=CLIP_LO, alpha=ROLL_ALPHA)
    predictions_final.loc[te_g.index, target_col] = pred

test_pred_long = test_df_tps[["forecast_id", "forecast_kst_dtm", "kpx_group"]].copy()
for group in [1, 2, 3]:
    mask = test_df_tps["kpx_group"] == group
    test_pred_long.loc[mask, "pred"] = predictions_final.loc[mask, f"kpx_group_{group}"]

submission_final = test_pred_long.pivot(index=["forecast_id", "forecast_kst_dtm"], columns="kpx_group", values="pred")
submission_final.columns = [f"kpx_group_{int(c)}" for c in submission_final.columns]
submission_final = submission_final.reset_index()
submission_final["forecast_kst_dtm"] = pd.to_datetime(submission_final["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")
submission_final = sample_submission[["forecast_id"]].merge(submission_final, on="forecast_id", how="left")

output_path = DATA_DIR / "simplified_nosnap_submit.csv"
submission_final.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved: {output_path}")

최종 학습:   0%|          | 0/3 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/풍력발전량예측/data/simplified_nosnap_submit.csv


In [18]:
# GBM 학습에 쓴 X_tr(마지막 fold 기준)로 피처 간 상관관계 확인
corr_matrix = X_tr.corr().abs()
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > 0.95:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

print(f"상관관계 0.95 초과 쌍: {len(high_corr_pairs)}개")
pd.DataFrame(high_corr_pairs, columns=["feat1", "feat2", "corr"]).sort_values("corr", ascending=False).head(30)

상관관계 0.95 초과 쌍: 160개


,feat1,feat2,corr
129,spa_g100_mean,sdG_ws100_mean,1.000000
29,ldaps_surface_0_avg_lsprate,ldaps_surface_0_ncpcp,0.999998
30,ldaps_surface_0_lssrate,ldaps_surface_0_snol,0.999767
42,gfs_heightAboveGround_80_u,gfs_heightAboveGround_100_100u,0.999674
73,gfs_ws_80m,gfs_ws_100m,0.999556
45,gfs_heightAboveGround_80_v,gfs_heightAboveGround_100_100v,0.999507
16,ldaps_heightAboveGround_2_t,sdL_t2_mean,0.999496
24,ldaps_surface_0_sp,sdL_sp_mean,0.999432
120,phy_ws100_cu,phy_rho_ws100_cu,0.999293
61,gfs_isobaricInhPa_850_t,sdG_t850_mean,0.999043


In [19]:
importance_df = pd.DataFrame({
    "feature": X_tr.columns,
    "importance": lgb.LGBMRegressor(**lgbm_params(500, 2, SEED)).fit(X_tr, y_tr).feature_importances_
}).sort_values("importance", ascending=False)

print("=== 상위 20개 ===")
print(importance_df.head(20))
print("\n=== 블록별 중요도 합계 ===")
importance_df["block"] = importance_df["feature"].apply(lambda f:
    "JMA" if f.startswith(("jgsm", "jmsm")) else
    "spd" if f.startswith(("sdL", "sdG")) else
    "phy" if f.startswith("phy") else
    "tmp" if f.startswith("tmp") else
    "spa" if f.startswith("spa") else
    "calendar" if f in ["month","day","hour","dayofweek","is_weekend","hour_sin","hour_cos","month_sin","month_cos"] else
    "v10_base")
print(importance_df.groupby("block")["importance"].sum().sort_values(ascending=False))

=== 상위 20개 ===
                         feature  importance
98               tmp_l10_r7_mean         357
327               sdG_ws850_mean         238
124                    phy_ws850         226
76       gfs_isobaricInhPa_500_v         196
137                      jmsm_ws         187
114                  tmp_day_std         184
112                 tmp_day_mean         182
67       gfs_isobaricInhPa_850_u         161
328                 sdG_ws850_g1         154
1                            day         146
19   ldaps_heightAboveGround_2_r         135
113                  tmp_day_max         135
56           gfs_surface_0_dlwrf         123
71       gfs_isobaricInhPa_700_u         120
314                  sdG_v100_g7         114
69       gfs_isobaricInhPa_850_r         112
75       gfs_isobaricInhPa_500_u         111
55           gfs_surface_0_dswrf         111
332                 sdG_ws850_g5         108
135                      jgsm_sp         105

=== 블록별 중요도 합계 ===
block
spd         74

In [ ]:
sub = pd.read_csv(DATA_DIR / "simplified_nosnap_submit.csv")
print(sub.shape, "| sample_submission:", sample_submission.shape)
for col in TARGET_COLS:
    print(col, "min:", sub[col].min(), "max:", sub[col].max(), "/ capacity:", CAPACITY_KWH[col])
print("forecast_id 순서 일치:", (sub["forecast_id"] == sample_submission["forecast_id"]).all())